# EDA: UIT-VSFC Vietnamese Sentiment Dataset
**Dataset:** UIT-VSFC (Vietnamese Students' Feedback Corpus)  
**Labels:** positive / neutral / negative  
**Splits:** train (11,426) / dev (1,583) / test (3,166)

## Research Questions
- **RQ1:** Khi neutral chỉ chiếm 4%, metric nào phản ánh chính xác nhất hiệu quả model?
- **RQ2:** Trên dataset câu ngắn như UIT-VSFC, PhoBERT có đạt F1 cao hơn đáng kể so với classical ML không?
- **RQ3:** Vietnamese tokenization ảnh hưởng thế nào đến TF-IDF + classical ML?

## 1. Load Data

In [1]:
import pandas as pd

train = pd.read_csv('../data/raw/train.csv')
dev   = pd.read_csv('../data/raw/dev.csv')
test  = pd.read_csv('../data/raw/test.csv')

print(f"Train: {len(train)} rows")
print(f"Dev:   {len(dev)} rows")
print(f"Test:  {len(test)} rows")

train.head(10)

Train: 11426 rows
Dev:   1583 rows
Test:  3166 rows


,Sentence,Topic,Sentiment,Encoded_topic,Encoded_sentiment
0,slide giáo trình đầy đủ .,program,positive,1,2
1,"nhiệt tình giảng dạy , gần gũi với sinh viên .",lecturer,positive,0,2
2,đi học đầy đủ full điểm chuyên cần .,program,negative,1,0
3,chưa áp dụng công nghệ thông tin và các thiết ...,lecturer,negative,0,0
4,"thầy giảng bài hay , có nhiều bài tập ví dụ ng...",lecturer,positive,0,2
5,"giảng viên đảm bảo thời gian lên lớp , tích cự...",lecturer,positive,0,2
6,"em sẽ nợ môn này , nhưng em sẽ học lại ở các h...",others,neutral,3,1
7,"thời lượng học quá dài , không đảm bảo tiếp th...",program,negative,1,0
8,"nội dung môn học có phần thiếu trọng tâm , hầu...",program,negative,1,0
9,cần nói rõ hơn bằng cách trình bày lên bảng th...,program,negative,1,0


## 2. Class Distribution

In [2]:
print(train['Sentiment'].value_counts())
print()
print((train['Sentiment'].value_counts(normalize=True) * 100).round(1))

Sentiment
positive    5643
negative    5325
neutral      458
Name: count, dtype: int64

Sentiment
positive    49.4
negative    46.6
neutral      4.0
Name: proportion, dtype: float64


## 3. Qualitative Inspection: Sample per Label

In [3]:
for label in ['positive', 'neutral', 'negative']:
    print(f"\n--- {label.upper()} ---")
    samples = train[train['Sentiment'] == label]['Sentence'].head(3).values
    for s in samples:
        print(f"  {s}")



--- POSITIVE ---
  slide giáo trình đầy đủ .
  nhiệt tình giảng dạy , gần gũi với sinh viên .
  thầy giảng bài hay , có nhiều bài tập ví dụ ngay trên lớp .

--- NEUTRAL ---
  em sẽ nợ môn này , nhưng em sẽ học lại ở các học kỳ kế tiếp .
  đang dạy thầy wzjwz208 đi qua nước ngoài giữa chừng , thầy wzjwz209 dạy thay .
  tạo ra sự cạnh tranh trong mỗi buổi thực hành .

--- NEGATIVE ---
  đi học đầy đủ full điểm chuyên cần .
  chưa áp dụng công nghệ thông tin và các thiết bị hỗ trợ cho việc giảng dạy .
  thời lượng học quá dài , không đảm bảo tiếp thu hiệu quả .


## 4. Text Length Analysis

In [4]:
train['word_count'] = train['Sentence'].str.split().str.len()

print(train.groupby('Sentiment')['word_count'].describe().round(1))

            count  mean   std  min  25%   50%   75%    max
Sentiment                                                 
negative   5325.0  16.9  12.1  2.0  9.0  13.0  21.0  159.0
neutral     458.0   9.8   8.4  2.0  5.0   7.0  12.0   79.0
positive   5643.0  12.2   7.1  2.0  8.0  10.0  15.0   95.0


## 5. Word Frequency per Label

In [5]:
#5. Từ xuất hiện nhiều nhất theo từng nhãn:
from collections import Counter

for label in ['positive', 'negative', 'neutral']:
    text = ' '.join(train[train['Sentiment'] == label]['Sentence'])
    words = text.split()
    top10 = Counter(words).most_common(10)
    print(f"\n--- {label.upper()} ---")
    for word, count in top10:
        print(f"  {word}: {count}")



--- POSITIVE ---
  .: 5443
  ,: 4092
  viên: 2555
  giảng: 2219
  tình: 2135
  dạy: 2135
  nhiệt: 1843
  thầy: 1831
  rất: 1607
  sinh: 1598

--- NEGATIVE ---
  .: 5145
  ,: 2651
  viên: 2178
  học: 1817
  không: 1814
  sinh: 1447
  giảng: 1445
  bài: 1300
  có: 1235
  thầy: 1170

--- NEUTRAL ---
  .: 421
  không: 116
  có: 101
  thầy: 94
  em: 92
  học: 85
  ,: 84
  viên: 70
  bài: 58
  kiến: 49


## 6. Preprocessing

In [6]:
import subprocess
subprocess.run(['pip', 'install', 'underthesea'], capture_output=True)
print("Done")

Done


In [7]:
import re
from underthesea import word_tokenize

def clean_text(text):
    text = str(text).lower()                        # bước 1: lowercase
    text = re.sub(r'[^\w\sÀ-ɏḀ-ỿ]', ' ', text)   # bước 2: xóa ký tự không phải chữ/số/dấu tiếng Việt
    text = re.sub(r'\s+', ' ', text).strip()        # bước 3: xóa khoảng trắng thừa
    return text

def tokenize_vi(text):
    return word_tokenize(text, format="text")       # underthesea tokenize

# Thử trên 1 câu trước
sample = "Nhiệt tình giảng dạy , gần gũi với sinh viên ."
print("Original: ", sample)
print("Cleaned:  ", clean_text(sample))
print("Tokenized:", tokenize_vi(clean_text(sample)))


Original:  Nhiệt tình giảng dạy , gần gũi với sinh viên .
Cleaned:   nhiệt tình giảng dạy gần gũi với sinh viên
Tokenized: nhiệt_tình giảng_dạy gần_gũi với sinh_viên


## 7. Áp dụng preprocessing lên toàn bộ dataset:

In [8]:
print("Đang xử lý train...")
before = len(train)
train = train.dropna(subset=['Sentence']).copy()
train['text_clean'] = train['Sentence'].apply(lambda x: tokenize_vi(clean_text(x)))
print(f"Train: {before} → {len(train)} rows")

print("\nĐang xử lý dev...")
dev['text_clean'] = dev['Sentence'].apply(lambda x: tokenize_vi(clean_text(x)))

print("\nĐang xử lý test...")
test['text_clean'] = test['Sentence'].apply(lambda x: tokenize_vi(clean_text(x)))

print("\nXong. Ví dụ kết quả:")
print(train[['Sentence', 'text_clean', 'Sentiment']].head(3).to_string())


Đang xử lý train...
Train: 11426 → 11426 rows

Đang xử lý dev...

Đang xử lý test...

Xong. Ví dụ kết quả:
                                         Sentence                                  text_clean Sentiment
0                       slide giáo trình đầy đủ .                     slide giáo_trình đầy_đủ  positive
1  nhiệt tình giảng dạy , gần gũi với sinh viên .  nhiệt_tình giảng_dạy gần_gũi với sinh_viên  positive
2            đi học đầy đủ full điểm chuyên cần .          đi học đầy_đủ full_điểm chuyên cần  negative


## 8. Lưu data đã xử lý:

In [9]:
import os
os.makedirs('../data/processed', exist_ok=True)

train.to_csv('../data/processed/train_processed.csv', index=False)
dev.to_csv('../data/processed/dev_processed.csv', index=False)
test.to_csv('../data/processed/test_processed.csv', index=False)

print("Đã lưu vào data/processed/")
print(f"  train_processed.csv: {len(train)} rows")
print(f"  dev_processed.csv:   {len(dev)} rows")
print(f"  test_processed.csv:  {len(test)} rows")

Đã lưu vào data/processed/
  train_processed.csv: 11426 rows
  dev_processed.csv:   1583 rows
  test_processed.csv:  3166 rows


## 9. SQL Analysis với DuckDB

In [10]:
import subprocess
subprocess.run(['pip', 'install', 'duckdb'], capture_output=True)

import duckdb
print("DuckDB version:", duckdb.__version__)

DuckDB version: 1.5.4


### Q1: Phân bố nhãn và tỉ lệ phần trăm (RQ1)


In [11]:
duckdb.sql("""
    SELECT
        Sentiment,
        COUNT(*)                                             AS so_cau,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS ti_le
    FROM train
    GROUP BY Sentiment
    ORDER BY so_cau DESC
""")


┌───────────┬────────┬────────┐
│ Sentiment │ so_cau │ ti_le  │
│  varchar  │ int64  │ double │
├───────────┼────────┼────────┤
│ positive  │   5643 │   49.4 │
│ negative  │   5325 │   46.6 │
│ neutral   │    458 │    4.0 │
└───────────┴────────┴────────┘

**Insight Q1:** neutral chỉ 4%, xác nhận class imbalance nghiêm trọng. Nếu model bỏ qua hoàn toàn lớp neutral vẫn đạt accuracy 96%, vì vậy cần F1-macro làm primary metric thay vì accuracy (RQ1).


### Q2: Thống kê độ dài câu theo nhãn (RQ2)


In [12]:
duckdb.sql("""
    SELECT
        Sentiment,
        ROUND(AVG(word_count), 1) AS avg_words,
        MEDIAN(word_count)        AS median_words,
        MAX(word_count)           AS max_words
    FROM train
    GROUP BY Sentiment
    ORDER BY avg_words DESC
""")


┌───────────┬───────────┬──────────────┬───────────┐
│ Sentiment │ avg_words │ median_words │ max_words │
│  varchar  │  double   │    double    │   int64   │
├───────────┼───────────┼──────────────┼───────────┤
│ negative  │      16.9 │         13.0 │       159 │
│ positive  │      12.2 │         10.0 │        95 │
│ neutral   │       9.8 │          7.0 │        79 │
└───────────┴───────────┴──────────────┴───────────┘

**Insight Q2:** Câu trong UIT-VSFC rất ngắn, median toàn dataset khoảng 10 từ. PhoBERT được thiết kế để hiểu ngữ cảnh dài, nhưng trên data ngắn như thế này chưa chắc đã vượt trội so với TF-IDF + classical ML. Đây là câu hỏi cần thực nghiệm trong RQ2.


### Q3: Từ ghép được nhận diện bởi underthesea theo nhãn (RQ3)


In [13]:
duckdb.sql("""
    SELECT
        Sentiment,
        token,
        COUNT(*) AS tan_suat
    FROM (
        SELECT Sentiment, UNNEST(STRING_SPLIT(text_clean, ' ')) AS token
        FROM train
    )
    WHERE token LIKE '%\_%' ESCAPE '\\'
    GROUP BY Sentiment, token
    HAVING COUNT(*) >= 10
    ORDER BY Sentiment, tan_suat DESC
    LIMIT 30
""")


┌───────────┬──────────────┬──────────┐
│ Sentiment │    token     │ tan_suat │
│  varchar  │   varchar    │  int64   │
├───────────┼──────────────┼──────────┤
│ negative  │ sinh_viên    │     1323 │
│ negative  │ thực_hành    │      695 │
│ negative  │ giảng_viên   │      680 │
│ negative  │ bài_tập      │      621 │
│ negative  │ môn_học      │      504 │
│ negative  │ kiến_thức    │      323 │
│ negative  │ thời_gian    │      320 │
│ negative  │ lý_thuyết    │      275 │
│ negative  │ nội_dung     │      269 │
│ negative  │ giảng_dạy    │      267 │
│    ·      │     ·        │       ·  │
│    ·      │     ·        │       ·  │
│    ·      │     ·        │       ·  │
│ negative  │ thông_báo    │      108 │
│ negative  │ thực_tế      │      106 │
│ negative  │ chất_lượng   │      103 │
│ negative  │ tiếp_thu     │       96 │
│ negative  │ một_số       │       93 │
│ negative  │ chương_trình │       92 │
│ negative  │ thường_xuyên │       90 │
│ negative  │ học_sinh     │       90 │


**Insight Q3:** underthesea nhận diện được nhiều từ ghép quan trọng như `sinh_viên`, `giảng_viên`, `bài_tập`, `kiến_thức`. Nếu dùng whitespace split, các từ này bị tách ra và mất nghĩa hoàn toàn, ảnh hưởng trực tiếp đến chất lượng feature của TF-IDF. Đây là lý do cần so sánh 2 cách tokenize trong RQ3.
